In [3]:
import geopandas as gpd
import pandas as pd
from rasterstats import zonal_stats
from pyprojroot import here

In [4]:
RAW_DATA_DIR = here() / "data" / "raw"
INTERIM_DATA_DIR = here() / "data" / "interim"

In [5]:
mun = gpd.read_file(RAW_DATA_DIR/"geography/municipalities/00mun.shp").to_crs(epsg=4326)

In [6]:
stats = zonal_stats(
    mun,
    RAW_DATA_DIR/"resources/wind_density_tif/MEX_power-density_100m.tif",
    stats=["mean", "max", "min", "std", "count"],
    all_touched=False,
)

In [7]:
wind = pd.DataFrame(stats)
wind.columns = ["wind_mean", "wind_max", "wind_min", "wind_std", "wind_pixel_count"]

In [8]:
result = pd.concat([
    mun[["CVEGEO", "CVE_ENT", "CVE_MUN", "NOMGEO"]].reset_index(drop=True),
    wind.reset_index(drop=True)
], axis=1)

In [9]:
# We verify that results are within reasonable bounds.
# wind_mean is the average possible energy output per square meter at an altitude of 100 meters.
print(f"Wind power density range: {result['wind_mean'].min():.1f} – {result['wind_mean'].max():.1f} W/m²")
print(f"Municipalities with data: {result['wind_mean'].notna().sum()} / {len(result)}")
print(result[["NOMGEO", "wind_mean", "wind_max"]].head(10))

Wind power density range: 0.0 – 783.1 W/m²
Municipalities with data: 2478 / 2478
                      NOMGEO  wind_mean    wind_max
0             Aguascalientes  40.689449  325.503418
1         San José de Gracia  15.157919  397.854736
2  San Francisco de los Romo  61.586220   96.556305
3                   Tepezalá  61.193790  285.488770
4            Rincón de Romos  22.748753  183.281464
5        Pabellón de Arteaga  45.162289  193.433090
6                      Cosío  44.792068  168.621277
7                Jesús María  31.029005  420.618164
8                   El Llano  56.833961  822.355957
9                   Calvillo  20.258993  526.653137


In [11]:
result = result[['CVEGEO', 'wind_mean']].copy()

In [14]:
result.to_csv(INTERIM_DATA_DIR/"wind_potential_by_municipality.csv", index=False, encoding="utf-8-sig")

print(f"Done. {len(result)} municipalities written.")
print(result.sample(10))

Done. 2478 municipalities written.
     CVEGEO  wind_mean
144   07048  48.318371
438   12044   1.093242
1315  20107  19.308235
941   17007   2.926627
837   16083   1.762386
234   08003  66.258720
1741  21049   0.709077
2448  32057  87.200218
1266  20520  11.859851
1790  21086   2.675996
